In [13]:
import os
import requests
from dotenv import load_dotenv
load_dotenv()



True

In [14]:


def jina_embedding_model(input_data) :
    url = "https://api.jina.ai/v1/embeddings"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {os.getenv("JINA_API_KEY")}"
    }
    data = {
        "model": "jina-embeddings-v2-base-en",
        "task": "separation",
        "input": input_data
    }
    response = requests.post(url, headers=headers, json=data)
    response.raise_for_status()
    return response
    

In [15]:
from pinecone import Pinecone, ServerlessSpec

In [16]:
EMBED_DIM = 768

In [17]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

In [18]:
INDEX_NAME = "jina-try-2"

In [19]:
# Create the index if it doesn't already exist
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBED_DIM,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(INDEX_NAME)

In [20]:
# ------------------------------------------------------------------
# 2. Chunking
# ------------------------------------------------------------------


from unstructured.partition.pdf import partition_pdf

pdf_path = "document.pdf"

chunks = partition_pdf(
    filename=pdf_path,
    strategy="hi_res",                 # better layout understanding
    infer_table_structure=True,        # keep tables structured
    extract_images_in_pdf=False,
    chunking_strategy="by_title",      # section-based chunking
    max_characters=1500,               # adjust based on your manual
    new_after_n_chars=1000,            # start new chunk before hitting max
    combine_text_under_n_chars=300,    # merge small pieces
    overlap=150
)



chunk_texts = [chunk.text for chunk in chunks]
chunk_page = [chunk.metadata.to_dict().get('page_number') for chunk in chunks]


No languages specified, defaulting to English.


In [21]:
# ------------------------------------------------------------------
# 3. Prepare and embed your documents
# ------------------------------------------------------------------



result = jina_embedding_model(chunk_texts)

data = sorted(result.json()["data"], key=lambda x: x["index"])
embeddings = [item["embedding"] for item in data]

In [22]:
# ------------------------------------------------------------------
# 4. Upsert (insert/update) vectors into Pinecone
# ------------------------------------------------------------------
vectors_to_upsert = [
    {
        "id": f"doc_{i}",
        "values": embeddings[i],
        "metadata": {"text": chunk_texts[i],
                     "page_number" : chunk_page[i]}  # store original text as metadata
    }
    for i in range(len(chunk_texts))
]

index.upsert(vectors=vectors_to_upsert, namespace="my-namespace")

UpsertResponse(upserted_count=29)

In [25]:
# ------------------------------------------------------------------
# 5. Query with a new sentence
# ------------------------------------------------------------------
query = "when i open my computer i see a blue screen of death also tell me how i should fix this issue"
query_embedding = jina_embedding_model([query]).json()['data'][0]['embedding']

results = index.query(
    vector=query_embedding,
    top_k=3,
    namespace="my-namespace",
    include_metadata=True
)

print("Query:", query)
print("Most similar documents:")
for match in results.matches:
    print(f"  - {match.metadata['text']}  (score: {match.score:.4f})")

Query: when i open my computer i see a blue screen of death also tell me how i should fix this issue
Most similar documents:
  - 3. Internet connection issues

Users may experience a loss of internet connectivity due to router problems, ISP issues, or wireless signal interference. Troubleshooting involves checking router settings, modem connections, or wireless configurations.

(1)

4. Blue screen of death (BSOD)

A blue screen error in Windows can indicate hardware or driver Troubleshooters may need to update drivers, check for hardware problems, or scan for malware. issues.

5. Software crashes or freezes

This can be caused by buggy applications, insufficient system resources, or conflicts between programs. Users may try updating the software, reinstalling it, or closing other resource-intensive applications.

6. Printer problems

Printing failures can be due to connectivity issues, outdated drivers, or paper jams. Troubleshooters should check printer connections, update drivers, an